In [ ]:
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
import math
import random
torch.manual_seed(0)
import matplotlib.pyplot as plt
plt.style.use('dark_background')
%matplotlib inline
import sys

torch.manual_seed(0)

In [ ]:
# Setup vocabulary
vocab = [
    "[PAD]", "[MASK]", "[UNK]",

    "I", "you", "we", "they",

    # Polysemous words
    "bear",        # animal / tolerate
    "run",         # move / operate
    "bank",        # river / finance
    "charge",      # legal / electrical

    # Nouns
    "river", "road", "field", "court", "battery",
    "money", "load", "power", "side", "shore", "swam",

    # Function words
    "to", "the", "a", "other", "across", "with", "in", "on", "of", "get", "cannot"
]

vocab_size = len(vocab)
print(f'Vocabulary size is {vocab_size}')

In [ ]:
# Training data
training_data = [
    (["I", "swam", "across", "the", "river", "to", "the", "other", "[MASK]"], "shore"),
    (["they", "went", "to", "the", "bank", "to", "get", "[MASK]"], "money"),
    (["I", "saw", "a", "bear", "in", "the", "[MASK]"], "field"),
    (["I", "cannot", "bear", "the", "[MASK]"], "load"),
    (["they", "run", "across", "the", "[MASK]"], "field"),
    (["the", "battery", "can", "run", "with", "[MASK]"], "power"),
    (["the", "court", "will", "charge", "them", "with", "[MASK]"], "money"),
    (["the", "battery", "has", "a", "charge", "of", "[MASK]"], "power"),
]

In [ ]:
sentence = training_data[0][0]
target = training_data[0][1]
idxtoword = {i:j for i,j in enumerate(vocab)}
wordtoidx = {j:i for i,j in enumerate(vocab)}
print(wordtoidx)
def encode(data):
    return torch.tensor([wordtoidx.get(i,wordtoidx["[UNK]"]) for i in data])

In [ ]:
class Transformer(nn.Module):
    def __init__(self,vocab_size,embedingsize):
        super().__init__()

        self.wordEmbedings = nn.Embedding(vocab_size,embedingsize)
        self.q_w = nn.Linear(embedingsize,embedingsize,bias=False)
        self.k_w = nn.Linear(embedingsize,embedingsize,bias=False)
        self.v_w = nn.Linear(embedingsize,embedingsize,bias=False)

        self.W =  nn.Linear(embedingsize,vocab_size,bias=False)

    def forward(self,sentenceidx,maskidx):
        data = self.wordEmbedings(sentenceidx)
        # print(data)
        Q = self.q_w(data)
        K = self.k_w(data)
        V = self.v_w(data)
        S = F.softmax((Q@K.T)/math.sqrt(Q.size(-1)),dim=-1)

        ouput = S@V

        z = self.W(ouput)

        return z[maskidx]

def lossFun(logets,targetIdx):
    probs = F.softmax(logets,dim=-1)
    # print(probs)
    eps = 1e-09
    return -torch.log(probs[targetIdx]+eps)

In [ ]:
vocab_size = len(vocab)
embedingsize = 8
epoch = 500
model = Transformer(vocab_size,embedingsize)
optimizer = torch.optim.Adam(model.parameters(),lr=0.05)
allLoss = []
for i in range(epoch):
    epochloss = 0.0
    random.shuffle(training_data)

    for sentence,target in training_data:
        sentenceidx = encode(sentence)
        # print(sentenceidx)
        maskidx = sentence.index("[MASK]")
        # print(maskidx)
        optimizer.zero_grad()
        targetIdx = wordtoidx[target]
        # print(targetIdx)

        logets = model(sentenceidx,maskidx)
        loss = lossFun(logets,targetIdx)
        loss.backward()
        optimizer.step()
        allLoss.append(loss.item())
        epochloss+=loss.item()
    if i%10==0:
        print(f"epoch {i} : {epochloss:4f}")


In [ ]:
no = np.random.randint(0,8)
print(np.random.randint(0,8))
sentence = training_data[no][0]
target = training_data[no][1]

print(sentence)
print(target)


sentenceidx = encode(sentence)
        # print(sentenceidx)
print(sentenceidx)
maskidx = sentence.index("[MASK]")
print(maskidx)

z = model(sentenceidx,maskidx)
probs = F.softmax(z,dim=-1)

print(f"pridiction : {idxtoword[torch.argmax(probs).item()]}")
